# Multi-q 3D RSM viewer (generic)

Loads a trained **joint multi-q SAXS-NAF** model and lets you explore it —
parametrized by a `QIndexedDataContainer` subclass and a results folder, so
the same notebook works for `frogbone`, `c4`, `c5`, or any future q-indexed
dataset with zero changes beyond the config cell below.

This supersedes the dataset-specific comparison logic that used to live in
one-off `compare_full_sweep_*.py` scripts — that logic now lives in
`smartt.saxs_naf.viz_multiq`, which this notebook just calls.

Sections:
1. **Config** — pick the dataset class + results folder.
2. **Load** the trained model.
3. **Baseline comparison** — table + log-log plot vs. independent
   per-shell baseline reconstructions (GK by default).
4. **3D cutaway sphere** — one voxel's full `I(q, theta, phi)`, rendered as
   a nested, cut-open sphere (radius = q).
5. **Interactive RSM slice viewer** — 2D cross-sections through
   `I(qx, qy, qz)` at an arbitrary voxel.
6. **Dual NAF-vs-baseline slice viewer** — same slices, side by side with
   the nearest independent baseline reconstruction.


In [ ]:
import sys, os, glob
sys.path.insert(0, "/myhome/smartt")
sys.path.insert(0, "/myhome/smartt/notebooks")
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

from smartt.saxs_naf.viz_multiq import (
    load_multiq_model, find_baseline_reconstructions,
    compare_multiq_vs_baselines, summarize_comparison, plot_baseline_vs_qres,
    run_full_comparison,
)
from smartt.saxs_naf.eval_multiq import sample_qshells_physical, fit_scale_trend
from mumott_plotting.rsm_3d_visualization import qshell_intensity_grid, interactive_cutaway_range
from mumott_plotting.rsm_slice_viewer import interactive_rsm_slice, rsm_slice, ORIENTATIONS, _embed_qxyz
from mumott_plotting.sh_visualization import generate_lm_list, evaluate_real_sh

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import plotly.io as pio
pio.renderers.default = "vscode"   # "notebook" relies on RequireJS, which VS Code's notebook webview doesn't expose


## 1. Config

Point these three lines at whatever dataset/run you want to explore — everything below is generic from here on.


In [ ]:
# --- CHANGE THESE THREE THINGS FOR A NEW DATASET/RUN ---
from smartt.data_containers.c5 import C5DataContainer as DATASET_CLS
RESULTS_DIR = "/myhome/data/smartt/shared/results/c5_benchmark/multiq_diagnostics"
BASELINE_METHOD = "mumott_gk"   # matches whatever method reconstruct_job.py ran for the baselines
# ---------------------------------------------------------

DC_TYPE = "main"
print(f"dataset={DATASET_CLS._NAME_PREFIX!r}  results_dir={RESULTS_DIR!r}  baseline={BASELINE_METHOD!r}")


## 2. Load the trained joint model

In [ ]:
model, meta = load_multiq_model(RESULTS_DIR, device=device)
print(f"Loaded {meta['source_path']}")
print(f"n_qshells={model.n_qshells}, ell_max={model.ell_max}, volume_shape={model.volume_shape}")


## 3. Baseline comparison

Per-shell agreement between the joint model and every completed independent baseline reconstruction found under `DATASET_CLS._CACHE_DIR_ROOT` — mean-c00-over-mask, full RSM correlation, and each method's own held-out-projection NRMSE (the real quality signal — needs no cross-method comparison at all).


In [ ]:
baseline_paths = find_baseline_reconstructions(DATASET_CLS, method=BASELINE_METHOD, dc_type=DC_TYPE)
print(f"found {len(baseline_paths)} {BASELINE_METHOD} baselines: {sorted(baseline_paths)}")

rows = compare_multiq_vs_baselines(DATASET_CLS, model, meta, baseline_paths=baseline_paths,
                                    method=BASELINE_METHOD, dc_type=DC_TYPE, device=device)
summary = summarize_comparison(rows)
display(pd.DataFrame(rows).set_index("qbin"))
summary


In [ ]:
plot_path = f"/myhome/smartt/notebooks/figures_wandb/{DATASET_CLS._NAME_PREFIX}_multiq_diagnostics/full_baseline_vs_qres.png"
fig, _ = plot_baseline_vs_qres(rows, dataset_name=DATASET_CLS._NAME_PREFIX,
                                baseline_label=BASELINE_METHOD.replace("_", " ").upper(),
                                save_path=plot_path)
plt.show()
print(f"saved to {plot_path}")


## 4. The 3D RSM cutaway sphere

Renders the full `I(q, theta, phi)` at one voxel as a solid, layered object
with a corner cut away — radius is literally q (the physical
reciprocal-space radial coordinate), and every real shell the model was
trained on is included at its own true radius (not a coarse subsample), so
the cut's walls show a genuine, continuous transition from the outer shell
down to the inner one.

Four controls: **top shell** (outer visible skin), **deep shell** (how far
the cut reveals inward), **cut frac** (wedge width), **log radius** (q on a
log scale, since it typically spans several decades).


In [ ]:
VOXEL = tuple(s // 2 for s in model.volume_shape)   # centre voxel by default -- change to explore others

all_q_sorted = sorted(meta["q_values"].keys(), key=lambda qb: meta["q_values"][qb])
q_phys_sorted = [meta["q_values"][qb] for qb in all_q_sorted]

with torch.no_grad():
    # Chunk over q to bound peak memory (evaluating all shells x full grid at once can OOM).
    CHUNK = 12
    voxel_rows = []
    for start in range(0, len(q_phys_sorted), CHUNK):
        chunk_q = q_phys_sorted[start:start + CHUNK]
        c = sample_qshells_physical(model, chunk_q, meta["log_q_min"], meta["log_q_max"],
                                     meta["target_scale_by_q"], meta["q_values"])
        voxel_rows.append(c[:, VOXEL[0], VOXEL[1], VOXEL[2], :].numpy())
    voxel_coeffs = np.concatenate(voxel_rows, axis=0)   # (Q, C)

q_arr = np.array(q_phys_sorted)
theta_grid, phi_grid, intensity = qshell_intensity_grid(voxel_coeffs, ell_max=model.ell_max, n_theta=40, n_phi=90)
interactive_cutaway_range(q_arr, intensity, theta_grid, phi_grid)


## 5. Interactive RSM slice viewer

Axial / coronal / sagittal 2D cross-sections through `I(qx, qy, qz)` at a
chosen voxel — every pixel is queried at its own exact, continuous
`(q, theta, phi)` through the trained field directly (not looked up on the
discrete trained q-shells and resampled), so the slice is smooth rather
than showing rings at the trained q-radii.


In [ ]:
interactive_rsm_slice(model, meta, VOXEL)


## 6. Dual NAF-vs-baseline slice viewer

Same slices as above, with a second row showing the nearest independent
baseline reconstruction (whatever `BASELINE_METHOD` found in section 3) at
the same voxel/orientation/position — unlike the NAF field, the baseline
has no continuous q-encoder, so each pixel is assigned the coefficients of
its *nearest* trained baseline q-shell (visible "rings" at the baseline's
discrete q-radii are the point of the contrast, not a bug).


In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
from matplotlib.lines import Line2D

qbins_sorted = sorted(meta["q_values"], key=lambda qb: meta["q_values"][qb])
q_sorted = [meta["q_values"][qb] for qb in qbins_sorted]
_c00_scale_trend = fit_scale_trend(meta["target_scale_by_q"], meta["q_values"])
_q_to_qbin = dict(zip(q_sorted, qbins_sorted))

_baseline_qbins = sorted(baseline_paths)
_baseline_q_sorted = np.array([meta["q_values"][qb] for qb in _baseline_qbins if qb in meta["q_values"]])
print(f"{BASELINE_METHOD} baseline available for {len(_baseline_qbins)}/{len(qbins_sorted)} trained qbins")

X_, Y_, Z_ = model.volume_shape
q_max_rsm = max(q_sorted)

_crosshair_handles = [Line2D([], [], color=c, lw=1.5) for c in ["limegreen", "tomato", "deepskyblue"]]
_crosshair_labels = ["x", "y", "z"]
_rsm_axis_labels = {"axial": ("q_x", "q_y", "q_z"),
                    "coronal": ("q_x", "q_z", "q_y"),
                    "sagittal": ("q_y", "q_z", "q_x")}


def _c00_volume(q_c00):
    q_norm = (np.log(q_c00) - meta["log_q_min"]) / (meta["log_q_max"] - meta["log_q_min"])
    with torch.no_grad():
        coeffs = model.forward_at_q(
            torch.tensor([q_norm], dtype=torch.float32, device=device)
        )[0].cpu().numpy()   # (X, Y, Z, C)
    return coeffs[..., 0] * _c00_scale_trend(q_c00)


def _baseline_c00_volume(q_c00):
    qb = _q_to_qbin.get(q_c00)
    if qb is None or qb not in baseline_paths:
        return None
    return np.load(baseline_paths[qb])[..., 0]


def _baseline_rsm_slice(voxel, orientation, position, q_max, n_pix):
    if not _baseline_qbins:
        return None, None, None
    coeffs_at_voxel = np.stack([
        np.load(baseline_paths[qb], mmap_mode="r")[voxel[0], voxel[1], voxel[2], :]
        for qb in _baseline_qbins
    ])   # (n_baseline, C)

    axis1 = np.linspace(-q_max, q_max, n_pix)
    axis2 = np.linspace(-q_max, q_max, n_pix)
    qx, qy, qz = _embed_qxyz(axis1, axis2, position, orientation)
    q = np.sqrt(qx ** 2 + qy ** 2 + qz ** 2)
    q_min_baseline = _baseline_q_sorted.min()
    out_of_range = (q < q_min_baseline) | (q > q_max)
    q_safe = np.clip(q, q_min_baseline, q_max)
    theta = np.arccos(np.clip(qz / q_safe, -1.0, 1.0))
    phi = np.arctan2(qy, qx)

    nearest_idx = np.abs(np.log(q_safe)[..., None] - np.log(_baseline_q_sorted)[None, None, :]).argmin(axis=-1)
    coeffs = coeffs_at_voxel[nearest_idx]   # (n_pix, n_pix, C)

    lm_list = generate_lm_list(model.ell_max)
    intensity = np.zeros((n_pix, n_pix), dtype=np.float64)
    for i, (l, m) in enumerate(lm_list):
        intensity += coeffs[..., i] * evaluate_real_sh(l, m, theta, phi)
    intensity[out_of_range] = np.nan
    return axis1, axis2, intensity


def _row(fig, gs, row, c00_volume, x, y, z, orientation, position, q_c00, vmin, vmax,
         rsm_axis1, rsm_axis2, rsm_intensity, row_label, c00_lo, c00_hi):
    ax_yz, ax_xz, ax_xy, ax_rsm = (fig.add_subplot(gs[row, i]) for i in range(4))
    kw = dict(cmap="inferno", vmin=c00_lo, vmax=c00_hi, aspect="equal", origin="lower")

    im = ax_yz.imshow(c00_volume[x, :, :].T, **kw)
    ax_yz.axvline(y, color="tomato", lw=0.9, alpha=0.85)
    ax_yz.axhline(z, color="deepskyblue", lw=0.9, alpha=0.85)
    ax_yz.set_title(f"YZ (x={x})"); ax_yz.set_xlabel("Y"); ax_yz.set_ylabel(f"{row_label}\nZ")

    ax_xz.imshow(c00_volume[:, y, :].T, **kw)
    ax_xz.axvline(x, color="limegreen", lw=0.9, alpha=0.85)
    ax_xz.axhline(z, color="deepskyblue", lw=0.9, alpha=0.85)
    ax_xz.set_title(f"XZ (y={y})"); ax_xz.set_xlabel("X"); ax_xz.set_ylabel("Z")

    ax_xy.imshow(c00_volume[:, :, z].T, **kw)
    ax_xy.axvline(x, color="limegreen", lw=0.9, alpha=0.85)
    ax_xy.axhline(y, color="tomato", lw=0.9, alpha=0.85)
    ax_xy.set_title(f"XY (z={z})"); ax_xy.set_xlabel("X"); ax_xy.set_ylabel("Y")
    ax_xy.legend(_crosshair_handles, _crosshair_labels, fontsize=7, loc="lower right",
                 framealpha=0.5, title="voxel")
    plt.colorbar(im, ax=[ax_yz, ax_xz, ax_xy], shrink=0.7, label=f"c00 (q={q_c00:.3g})")

    disp = np.log10(np.clip(rsm_intensity, 1e-300, None))
    im2 = ax_rsm.imshow(disp, origin="lower",
                         extent=[rsm_axis1.min(), rsm_axis1.max(), rsm_axis2.min(), rsm_axis2.max()],
                         cmap="inferno", aspect="equal", vmin=vmin, vmax=vmax)
    xlabel, ylabel, fixed_label = _rsm_axis_labels[orientation]
    ax_rsm.set_xlabel(xlabel); ax_rsm.set_ylabel(ylabel)
    ax_rsm.set_title(f"RSM {orientation} ({fixed_label}={position:.3g})")
    plt.colorbar(im2, ax=ax_rsm, shrink=0.7, label="log10 I(q)")


def _explore(x, y, z, orientation, position, q_c00, vmin, vmax):
    naf_c00 = _c00_volume(q_c00)
    naf_axis1, naf_axis2, naf_intensity = rsm_slice(model, meta, (x, y, z), orientation=orientation,
                                                     position=position, q_max=q_max_rsm, n_pix=121)

    base_c00 = _baseline_c00_volume(q_c00)
    base_axis1, base_axis2, base_intensity = _baseline_rsm_slice((x, y, z), orientation, position,
                                                                   q_max_rsm, n_pix=121)

    n_rows = 2 if base_c00 is not None else 1
    fig = plt.figure(figsize=(18, 4.4 * n_rows))
    gs = fig.add_gridspec(n_rows, 4, wspace=0.4, hspace=0.5)

    if base_c00 is not None:
        c00_lo, c00_hi = np.nanpercentile(np.stack([naf_c00, base_c00]), [1, 99])
    else:
        c00_lo, c00_hi = np.percentile(naf_c00, [1, 99])

    _row(fig, gs, 0, naf_c00, x, y, z, orientation, position, q_c00, vmin, vmax,
         naf_axis1, naf_axis2, naf_intensity, "NAF (qres)", c00_lo, c00_hi)
    if base_c00 is not None:
        _row(fig, gs, 1, base_c00, x, y, z, orientation, position, q_c00, vmin, vmax,
             base_axis1, base_axis2, base_intensity, f"Baseline\n({BASELINE_METHOD})", c00_lo, c00_hi)
    else:
        fig.text(0.5, 0.02, f"No independent baseline for qbin {_q_to_qbin.get(q_c00)}",
                  ha="center", fontsize=10, color="gray")

    plt.suptitle(f"voxel ({x}, {y}, {z}) — NAF (qres) vs. independent {BASELINE_METHOD} baseline", fontsize=12)
    plt.show()


_axis1_0, _axis2_0, _intensity_0 = rsm_slice(model, meta, VOXEL, orientation="axial",
                                              position=0.0, q_max=q_max_rsm, n_pix=121)
_disp_0 = np.log10(np.clip(_intensity_0, 1e-300, None))
_vmin_default, _vmax_default = np.nanpercentile(_disp_0, [1, 99])
_disp_lo, _disp_hi = np.nanmin(_disp_0), np.nanmax(_disp_0)
_disp_pad = 0.1 * (_disp_hi - _disp_lo)

interact(
    _explore,
    x=widgets.IntSlider(min=0, max=X_ - 1, step=1, value=VOXEL[0], description="x", continuous_update=False),
    y=widgets.IntSlider(min=0, max=Y_ - 1, step=1, value=VOXEL[1], description="y", continuous_update=False),
    z=widgets.IntSlider(min=0, max=Z_ - 1, step=1, value=VOXEL[2], description="z", continuous_update=False),
    orientation=widgets.ToggleButtons(options=list(ORIENTATIONS), value="axial", description="orientation"),
    position=widgets.FloatSlider(min=-q_max_rsm, max=q_max_rsm, step=q_max_rsm / 100, value=0.0,
                                  description="q position", continuous_update=False, readout_format=".4f"),
    q_c00=widgets.SelectionSlider(options=[(f"{q:.4g}", q) for q in q_sorted], value=q_sorted[0],
                                   description="q-shell (c00)", continuous_update=False),
    vmin=widgets.FloatSlider(min=_disp_lo - _disp_pad, max=_disp_hi + _disp_pad, step=0.05,
                              value=_vmin_default, description="vmin (log10 I)", continuous_update=False,
                              readout_format=".2f"),
    vmax=widgets.FloatSlider(min=_disp_lo - _disp_pad, max=_disp_hi + _disp_pad, step=0.05,
                              value=_vmax_default, description="vmax (log10 I)", continuous_update=False,
                              readout_format=".2f"),
)
